In [ ]:
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
from maskedtensor import masked_tensor

from Wordle import WordleEnv
from models.DQN import DQN
from models.ActorCritic import Actor, Critic
from ReplayMemory import ReplayMemory
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))

# Initialize the environment (input subset size if necessary)
size = None
env = WordleEnv(subset_size=size) 

# set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

# if GPU is to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

torch.set_default_device(device)

## Actor Critic

In [ ]:
# Actor and critic instances for the protagonist
actor_p = Actor(env.state_size, env.action_size).to(device)
critic_p = Critic(env.state_size, env.action_size).to(device)

# Actor and critic for the antagonist
actor_a = Actor(env.state_size, env.action_size).to(device)
critic_a = Critic(env.state_size, env.action_size).to(device)

# Actor and critic for the teacher
actor_t = Actor(8, env.action_size).to(device)
critic_t = Critic(8, env.action_size).to(device)

def select_action_actor(actor, state, available_actions, action_size):
    # Create a mask tensor for previously chosen actions
    mask = torch.full((action_size,), -1e-9, device=device)
    for idx in available_actions:
        mask[idx] = 0

    # Add the mask to the actor output and sample from the distribution
    actor_output = actor(state)
    masked_output = actor_output + mask
    masked_distribution = Categorical(logits=masked_output)
    return masked_distribution.sample().view(1, 1)
        
def select_word(actor_t):
    """
    Selects a word from the distribution.
    """
    z = torch.randn(1, 48, device=device) # As per PAIRED paper
    teacher_output = actor_t(z)
    distribution = Categorical(logits=teacher_output)
    return env.words[distribution.sample().view(1, 1)]

def eval_choice(critic_t, state, regret):
    # Evaluates the teacher's choice based on regret
    log_probs = []
    values = []
    rewards = regret
    masks = []

    value = critic_t[state]
    if value.dim() > 1:
        values.append(value.squeeze(0))

In [ ]:
def run_episode(actor, critic, optimizer_actor, optimizer_critic, state): 
    """
    Runs one episode for the protagonist/antagonist
    """
    log_probs = []
    values = []
    rewards = []
    masks = []
    
    for i in count():
        actor_output, value = actor(state), critic(state)
        mask = torch.full((env.action_size,), -1e-9, device=device)
        for idx in env.available_actions:
            mask[idx] = 0.0

        masked_output = actor_output + mask
        dist = Categorical(logits=masked_output)
        action = dist.sample().view(1, 1)
        
        next_state, reward, done, _ = env.step(action.item())
        next_state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)
    
        log_prob = dist.log_prob(action).unsqueeze(1)
    
        log_probs.append(log_prob)
        if value.dim() > 1:
            values.append(value.squeeze(0))
        else:
            values.append(value)
        rewards.append(torch.tensor([reward], dtype=torch.float, device=device))
        masks.append(torch.tensor([1-done], dtype=torch.float, device=device))
    
        state = next_state
    
        if done:
            if episode % 1000 == 0:
                print(f"Episode: {episode}/{num_episodes}, Attempts: {env.attempts}, Reward: {reward}")
            break
    returns = []
    reward = 0

    for r, m in  zip(reversed(rewards), reversed(masks)):
        reward = r + 0.99 * reward * m # Setting gamma to 0.99 for discounted reward
        returns.insert(0, reward)

    returns = torch.cat(returns).detach()
    if values:
        values = torch.cat(values, dim=0).unsqueeze(1)
    else:
        # If values is empty, initialize it with a dummy tensor to avoid errors
        values = torch.zeros(1, 1, device=device, requires_grad=True)    
    log_probs = torch.cat(log_probs)
    

    advantage = returns - values
    
    optimizer_actor.zero_grad()
    optimizer_critic.zero_grad()
    
    actor_loss = -(log_probs * advantage.detach()).mean()
    critic_loss = advantage.pow(2).mean()

    actor_loss.backward()
    critic_loss.backward()
    optimizer_actor.step()
    optimizer_critic.step()

    return returns

In [ ]:
average_reward = 0
num_episodes = 10000

optimizer_actor_p = optim.Adam(actor_p.parameters(), lr=0.001)
optimizer_critic_p = optim.Adam(critic_p.parameters(), lr=0.001)

optimizer_actor_a = optim.Adam(actor_a.parameters(), lr=0.001)
optimizer_critic_a = optim.Adam(critic_a.parameters(), lr=0.001)

optimizer_actor_t = optim.Adam(actor_t.parameters(), lr=0.001)
optimizer_critic_t = optim.Adam(critic_t.parameters(), lr=0.001)

for episode in range(num_episodes):
    print(episode)
    z = torch.randn(1, 8, device=device) # As per PAIRED paper
    teacher_output = actor_t(z)
    distribution = Categorical(logits=teacher_output)
    teacher_action =  distribution.sample().view(1, 1)
    target = env.words[teacher_action]
    log_prob = distribution.log_prob(teacher_action)
    
    state = env.reset()
    env.target_word = target
    state = torch.FloatTensor(state).to(device)
    reward_p = run_episode(actor_p, critic_p, optimizer_actor_p, optimizer_critic_p, state=state)
    
    state = env.reset()
    env.target_word = target
    state = torch.FloatTensor(state).to(device)
    reward_a = run_episode(actor_a, critic_a, optimizer_actor_a, optimizer_critic_a, state=state)

    regret = reward_a.sum() - reward_p.sum()
    value = critic_t(z)
    advantage = regret - value
        
    optimizer_actor_t.zero_grad()
    optimizer_critic_t.zero_grad()
    
    actor_t_loss = -(log_prob * advantage.detach())
    critic_t_loss = advantage.pow(2)
    
    actor_t_loss.backward()
    critic_t_loss.backward()
    
    optimizer_actor_t.step()
    optimizer_critic_t.step()    

In [ ]:
total_attempts = 0
correct_guesses = 0

no_test_trials = 1000

eps_threshold = 1e-11 #For only exploration

for episode in range(no_test_trials):
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    for t in count():
        action = select_action_actor(state, env.available_actions, env.action_size)
        observation, reward, done, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        
        if done:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        state = next_state
        total_attempts += 1

        if done:
            if(reward[0] == 10):
                correct_guesses += 1
            break

success_rate = correct_guesses / (no_test_trials)
average_attempts = total_attempts / (no_test_trials)

print(f"Trials: {no_test_trials}, Success rate: {success_rate:.2f}, Average number of attempts: {average_attempts:.2f}")

In [ ]:
total_attempts = 0
correct_guesses = 0

no_test_trials = 1000

eps_threshold = 1e-11 #For only exploration

for episode in range(no_test_trials):
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    
    action = torch.tensor([[345]], device=device, dtype=torch.long) #Salet start.
    for t in count():
        observation, reward, done, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        
        if done:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        state = next_state
        total_attempts += 1

        if done:
            if(reward[0] == 10):
                correct_guesses += 1
            break
        action = select_action_actor(state, env.available_actions, env.action_size)

success_rate = correct_guesses / (no_test_trials)
average_attempts = total_attempts / (no_test_trials)

print(f"Trials with SALET start: {no_test_trials}, Success rate: {success_rate:.2f}, Average number of attempts: {average_attempts:.2f}")

In [ ]:
state = env.reset()
env.target_word = 'OTHER'
state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
action = torch.tensor([[344]], device=device, dtype=torch.long) #Salet start.

for t in count():
    observation, reward, done, _ = env.step(action.item())
    reward = torch.tensor([reward], device=device)
    env.render()
    if done:
        next_state = None
    else:
        next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
        
    if done:
        break
    action = select_action_actor(state, env.available_actions, env.action_size)
    state = next_state